In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [2]:
df = pd.read_csv('train.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
sns.countplot(x='Survived', data=df)
plt.title('Sobreviventes do Titanic')
plt.xlabel('Sobreviveu (0=Não, 1=Sim)')
plt.ylabel('Quantidade')
plt.show()

In [ ]:
sns.countplot(x='Survived', hue='Pclass', data=df)
plt.title('Sobreviventes por sexo')
plt.xlabel('Sobreviveu (0=Não, 1=Sim)')
plt.ylabel('Quantidade')
plt.show()

In [ ]:
sobreviventes_homens = df[(df['Survived'] == 1) & (df['Sex'] == 'male')]
print(f"Total de homens sobreviventes: {len(sobreviventes_homens)}")
print(f"Média de idade dos homens sobreviventes: {sobreviventes_homens['Age'].mean():.1f} anos")
print(f"Homens sobreviventes abaixo de 18 anos: {len(sobreviventes_homens[sobreviventes_homens['Age'] < 18])}")

In [ ]:
sobrevivencia_por_classe = df.groupby('Pclass')['Survived'].mean() * 100
print("Taxa de sobrevivência por classe:")
for classe, taxa in sobrevivencia_por_classe.items():
    print(f"Classe {classe}: {taxa:.1f}%")

In [ ]:
df.isnull().sum()

In [ ]:
print(df['Embarked'].value_counts())

In [ ]:
df['Embarked'] = df['Embarked'].fillna('S')
print(f"Valores faltantes em Embarked: {df['Embarked'].isnull().sum()}")

In [ ]:
mediana_idade = df['Age'].median()
print(f"Mediana de Idade: {mediana_idade}")
df['Age'] = df['Age'].fillna(mediana_idade)
print(f"Valores faltantes em idade: {df['Embarked'].isnull().sum()}")

In [ ]:
df = df.drop(columns=['Cabin'])
print(f"Colunas restantes: {df.columns.tolist()}")

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df['Age'], bins=30, kde=True)
plt.title('Distribuição de idade dos passageiros')
plt.xlabel('Idade')
plt.ylabel('Quantidade')

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(data=df[df['Survived']==0], x='Age', bins=30, color='red', alpha=0.5, label='Não sobreviveu')
sns.histplot(data=df[df['Survived']==1], x='Age', bins=30, color='green', alpha=0.5, label='Sobreviveu')
plt.title('Distribuição de idades por sobrevivência')
plt.xlabel('Idade')
plt.ylabel('Quantidade')
plt.legend()
plt.show()

In [ ]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
print(df['Title'].value_counts())

In [ ]:
df['Title'] = df['Title'].replace(
    ['Dr', 'Rev', 'Major', 'Col', 'Countess', 'Capt', 'Ms', 'Sir', 'Lady', 'Don', 'Jonkheer', 'Mme'], 'Rare'
)
df['Title'] = df['Title'].replace('Mlle', 'Miss')
print(df['Title'].value_counts())

In [ ]:
taxa_sobrevivencia_titulo = df.groupby('Title')['Survived'].mean() * 100
print('Taxa de sobrevivência por título')
for titulo, taxa in taxa_sobrevivencia_titulo.items():
    print(f"{titulo}: {taxa:.1f}%")

In [ ]:
df['FaixaEtaria'] = pd.cut(df['Age'],
                            bins=[0, 12, 18, 35, 60, 100],
                            labels=['Criança', 'Adolescente', 'Adulto Jovem', 'Adulto', 'Idoso'])
print(df['FaixaEtaria'].value_counts())

In [ ]:
taxa_sobrevivencia_idade = df.groupby('FaixaEtaria', observed=True)['Survived'].mean() * 100
print("Taxa de sobrevivência por faixa etária")
for faixa, taxa in taxa_sobrevivencia_idade.items():
    print(f"{faixa:} {taxa:.1f}%")

In [ ]:
df['Sex'] = df['Sex'].map({'male':0,'female':1})
df['Title'] = df['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})
df['FaixaEtaria'] = df['FaixaEtaria'].map({'Criança': 0, 'Adolescente': 1, 'Adulto Jovem': 2, 'Adulto': 3, 'Idoso': 4})


print(df[['Sex', 'Title', 'FaixaEtaria']].head())

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Title', 'FaixaEtaria']

X = df[features]
y = df['Survived']

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Treino: {X_train.shape}")
print(f"Teste: {X_test.shape}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

print("Modelo treinado com sucesso!")

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = modelo.predict(X_test)
acuracia = accuracy_score(y_test, y_pred)
print(f"Acurácia do modelo: {acuracia*100:.1f}%")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)
acuracia_rf = accuracy_score(y_test, y_pred_rf)
print(f"Acurácia Random Forest: {acuracia_rf*100:.1f}%")

In [ ]:
importancias = pd.DataFrame({
    'Feature': features,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=False)

print(importancias)

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=importancias, x='Importancia', y='Feature')
plt.title('Importância das Features')
plt.xlabel('Importância')
plt.ylabel('Feature')
plt.show()

In [ ]:
df_test = pd.read_csv('test.csv')
df_test.head()

In [ ]:
# Limpeza do test.csv
df_test['Embarked'] = df_test['Embarked'].fillna('S')
df_test['Age'] = df_test['Age'].fillna(df_test['Age'].median())
df_test['Fare'] = df_test['Fare'].fillna(df_test['Fare'].median())
df_test = df_test.drop(columns=['Cabin'])

# Feature Engineering
df_test['Title'] = df_test['Name'].str.extract(r' ([A-Za-z]+)\.')
df_test['Title'] = df_test['Title'].replace(['Dr', 'Rev', 'Major', 'Col', 'Capt', 'Countess', 'Sir', 'Lady', 'Don', 'Jonkheer', 'Mme', 'Ms'], 'Rare')
df_test['Title'] = df_test['Title'].replace('Mlle', 'Miss')

# Converter para números
df_test['Sex'] = df_test['Sex'].map({'male': 0, 'female': 1})
df_test['Title'] = df_test['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})
df_test['FaixaEtaria'] = pd.cut(df_test['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Criança', 'Adolescente', 'Adulto Jovem', 'Adulto', 'Idoso'])
df_test['FaixaEtaria'] = df_test['FaixaEtaria'].map({'Criança': 0, 'Adolescente': 1, 'Adulto Jovem': 2, 'Adulto': 3, 'Idoso': 4})

print("Test.csv preparado!")
print(df_test.isnull().sum())

In [ ]:
df_test['Title'] = df_test['Title'].fillna(4)
print(f"Valores faltantes em Title: {df_test['Title'].isnull().sum()}")

In [ ]:
X_submission = df_test[features]
previsoes = modelo_rf.predict(X_submission)

submission = pd.DataFrame({
    'PassengerId': df_test['PassengerId'],
    'Survived': previsoes
})

submission.to_csv('submission.csv', index=False)
print("Arquivo criado com sucesso!")
print(submission.head(10))